In [1]:
import numpy as np
import pandas as pd
from scipy import stats

# Simulate training data distribution

In [2]:
np.random.seed(42)
train_data = {
    'transaction_amount': np.random.normal(45, 20, 1000),
    'transaction_hour': np.random.randint(0, 24, 1000),
    'account_age_days': np.random.normal(365, 100, 1000)
}
train_df = pd.DataFrame(train_data)

print("Training data statistics:")
print(train_df.describe().round(2))

Training data statistics:
       transaction_amount  transaction_hour  account_age_days
count             1000.00           1000.00           1000.00
mean                45.39             11.52            366.43
std                 19.58              6.97             97.55
min                -19.83              0.00             63.05
25%                 32.05              5.00            300.80
50%                 45.51             11.50            364.97
75%                 57.96             18.00            432.82
max                122.05             23.00            684.31


In [3]:
# Scenario 1 — No drift (normal production data)
production_normal = {
    'transaction_amount': np.random.normal(46, 21, 500),  # barely different
    'transaction_hour': np.random.randint(0, 24, 500),
    'account_age_days': np.random.normal(370, 102, 500)
}
production_normal_df = pd.DataFrame(production_normal)

In [4]:
# Scenario 2 — Severe drift (new payment method launched)
production_drift = {
    'transaction_amount': np.random.normal(120, 80, 500),  # completely different
    'transaction_hour': np.random.randint(0, 24, 500),
    'account_age_days': np.random.normal(200, 50, 500)   # newer accounts
}
production_drift_df = pd.DataFrame(production_drift)

# KS test drift detector

In [6]:
def detect_drift(train_df,production_df,threshold=0.05):
    results = []
    
    for columns in train_df.columns:
        # KS test compares two distributions
        # returns statistic (how different) and p_value (is it significant?)
        statistic , p_value = stats.ks_2samp(
            train_df[columns],
            production_df[columns]
        )

        drift_detected = p_value < threshold

        results.append({
            'feature': columns,
            'ks_statistic': round(statistic,4),
            'p_value': round(p_value,4),
            'drift_detected': drift_detected
        })
    return pd.DataFrame(results)

In [7]:
print("=== Scenario 1: Normal Production Data ===")
print(detect_drift(train_df, production_normal_df))

print("\n=== Scenario 2: Drifted Production Data ===")
print(detect_drift(train_df, production_drift_df))

=== Scenario 1: Normal Production Data ===
              feature  ks_statistic  p_value  drift_detected
0  transaction_amount         0.049   0.3962           False
1    transaction_hour         0.040   0.6556           False
2    account_age_days         0.047   0.4488           False

=== Scenario 2: Drifted Production Data ===
              feature  ks_statistic  p_value  drift_detected
0  transaction_amount         0.670   0.0000            True
1    transaction_hour         0.053   0.3029           False
2    account_age_days         0.783   0.0000            True


# alerting

In [8]:
def check_and_alert(train_df,production_df , threshold=0.05):
    results = detect_drift(train_df,production_df,threshold)

    drifted_features = results[results['drift_detected']==True]

    if len(drifted_features) == 0:
        print("✅ No drift detected — model inputs are stable")
    else:
        print(f"🚨 DRIFT ALERT — {len(drifted_features)} feature(s) drifted:")
        for _, row in drifted_features.iterrows():
            print(f"   {row['feature']}: KS={row['ks_statistic']}, "
                  f"p={row['p_value']} — RETRAIN NEEDED")

In [9]:
print("=== Scenario 1 ===")
check_and_alert(train_df, production_normal_df)

print("\n=== Scenario 2 ===")
check_and_alert(train_df, production_drift_df)

=== Scenario 1 ===
✅ No drift detected — model inputs are stable

=== Scenario 2 ===
🚨 DRIFT ALERT — 2 feature(s) drifted:
   transaction_amount: KS=0.67, p=0.0 — RETRAIN NEEDED
   account_age_days: KS=0.783, p=0.0 — RETRAIN NEEDED


In [ ]:
import schedule
import time

def run_drift_check():
    # In production this would:
    # 1. Load training data from your feature store
    # 2. Load last 24 hours of production data
    # 3. Run detect_drift()
    # 4. Send alert to Slack/PagerDuty if drift detected
    print(f"Running drift check at {pd.Timestamp.now()}")
    check_and_alert(train_df, production_drift_df)

# Run drift check every 24 hours
schedule.every(24).hours.do(run_drift_check)

# Run once immediately to test
run_drift_check()

## Summary

**What I built:** Statistical drift detector using KS test —
detects when production data distribution shifts from training data.

**Key concepts:**
- Data drift: same columns, shifted distributions — no errors, silent failure
- KS test: compares two distributions statistically
  - statistic: how different (0=identical, 1=completely different)
  - p_value: is it significant? p < 0.05 = alert
- Alert threshold p=0.05: below this = statistically significant drift
- Alert fatigue: alerting on every tiny shift = worse than no alerts

**Results:**
- Scenario 1 (normal): all p_values > 0.05, no alerts
- Scenario 2 (drifted): transaction_amount KS=0.670, account_age KS=0.783
- transaction_hour: no drift — same uniform distribution both scenarios

**Why drift kills models silently:**
- No errors, no crashes, Prometheus shows normal latency
- Model makes confident wrong predictions
- You find out when a user complains — too late

**Production schedule:**
- Run drift check every 24 hours
- Compare last 24h production data vs training data
- Alert to Slack/PagerDuty if p < 0.05

**Next:** DVC data versioning — Day 47